### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
sys.path.append('./utils')

### Random seed for reproducibility

In [2]:
import torch
import random
import numpy as np
#import multiprocessing as mp
#mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc
import svg_constraints 
from svg_processor import SVGSanitizer, SVGProcessor

class Model:
    
    def __init__(self):

        self.model_path="./lora/Llama_32_1B_Instruct_FFT_fp16_s2000_i1000_msl_2048"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            gpu_memory_utilization=0.85,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )
       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, clean_svg, self.default_svg)
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 04-24 22:51:09 [__init__.py:239] Automatically detected platform cuda.


In [4]:
model=Model()

WARNING 04-24 22:51:10 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-24 22:51:14 [config.py:585] This model supports multiple tasks: {'reward', 'generate', 'classify', 'score', 'embed'}. Defaulting to 'generate'.
INFO 04-24 22:51:14 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-24 22:51:15 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Llama_32_1B_Instruct_FFT_fp16_s2000_i1000_msl_2048', speculative_config=None, tokenizer='./lora/Llama_32_1B_Instruct_FFT_fp16_s2000_i1000_msl_2048', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConf

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 04-24 22:51:17 [loader.py:447] Loading weights took 0.87 seconds
INFO 04-24 22:51:17 [gpu_model_runner.py:1186] Model loading took 2.3185 GB and 1.015668 seconds
INFO 04-24 22:51:22 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/30c0a68c5c/rank_0_0 for vLLM's torch.compile
INFO 04-24 22:51:22 [backends.py:425] Dynamo bytecode transform time: 4.05 s


[rank0]:W0424 22:51:22.894000 56869 site-packages/torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode


INFO 04-24 22:51:24 [backends.py:132] Cache the graph of shape None for later use
INFO 04-24 22:51:33 [backends.py:144] Compiling a graph for general shape takes 11.60 s
INFO 04-24 22:51:39 [monitor.py:33] torch.compile takes 15.65 s in total
INFO 04-24 22:51:40 [kv_cache_utils.py:566] GPU KV cache size: 182,928 tokens
INFO 04-24 22:51:40 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 178.64x
INFO 04-24 22:51:53 [gpu_model_runner.py:1534] Graph capturing finished in 14 secs, took 0.31 GiB
INFO 04-24 22:51:53 [core.py:151] init engine (profile, create kv cache, warmup model) took 35.74 seconds


In [24]:
model.predict(['sun rising in the east','A golden goose with a fish'])

AttributeError: 'Model' object has no attribute 'model'

In [6]:
#print(tmp[0])

In [7]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test_vqa.csv',header=[0])
print(df.shape)
df.head(2)

(75, 7)


,description,gpt_svg,gpt_score_sl,response,vqa_pair,response_2,gpt_svg_2
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [8]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [9]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 12
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


Batch prediction:   0%|                                   | 0/7 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/12 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   8%| | 1/12 [00:00<00:02,  4.14it/s, est. speed input: 252.3
cessed prompts:  17%|▏| 2/12 [00:00<00:01,  5.22it/s, est. speed input: 306.2
cessed prompts:  25%|▎| 3/12 [00:01<00:03,  2.61it/s, est. speed input: 183.2
cessed prompts:  33%|▎| 4/12 [00:01<00:02,  3.17it/s, est. speed input: 200.5
cessed prompts:  42%|▍| 5/12 [00:01<00:02,  2.75it/s, est. speed input: 183.2
cessed prompts:  50%|▌| 6/12 [00:02<00:02,  2.84it/s, est. speed input: 184.3
cessed prompts:  67%|▋| 8/12 [00:02<00:00,  4.84it/s, est. speed input: 232.3
cessed prompts:  75%|▊| 9/12 [00:02<00:00,  5.35it/s, est. speed input: 245.4
cessed prompts:  83%|▊| 10/12 [00:02<00:00,  5.25it/s, est. speed input: 250.
cessed prompts:  92%|▉| 11/12 [00:02<00:00,  5.17it/s, est. speed input: 254.
Processed prompts: 100%|█| 12/12 [00:02<00:00,  4.08it/s, est

In [10]:
df['svg_3']=results

In [11]:
model.close_model()

In [12]:
from siglip_class import SVGMetricEvaluator
from aesthetic_evaluator import AestheticEvaluator

In [13]:
#SigLip Score
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

100%|███████████████████████████████████████████| 75/75 [00:04<00:00, 16.01it/s]


In [14]:
#SigLip Score
tqdm.pandas()
aes_eval = AestheticEvaluator()
df['aes_score_3'] = df.progress_apply(lambda row: aes_eval.get_score(row['gpt_svg_2']), axis=1)

100%|███████████████████████████████████████████| 75/75 [00:09<00:00,  7.58it/s]


In [15]:
#combined score
df['combined_score_3'] = (df['svg_score_3']+df['svg_score_3']+df['aes_score_3'])/3

In [16]:
print('mean_svg_score:',df['svg_score_3'].mean(),'mean_aes_score:',df['aes_score_3'].mean(),'combined_score:',df['combined_score_3'].mean())

mean_svg_score: 0.03092420762564351 mean_aes_score: 0.48672293345133455 combined_score: 0.18285711623420722


In [17]:
default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
df_default_svg=df[df['svg_3']==default_svg]
print('default_svg_count:',df_default_svg.shape[0])
print('default_svg_score_mean:',df_default_svg['svg_score_3'].mean(),'default_aes_score_mean:',df_default_svg['aes_score_3'].mean(),\
     'combined_score:',df_default_svg['combined_score_3'].mean())

default_svg_count: 0
default_svg_score_mean: nan default_aes_score_mean: nan combined_score: nan


In [18]:
df_non_default_svg=df[df['svg_3']!=default_svg]
print('non-default_svg_count:',df_non_default_svg.shape[0])
print('non-default_svg_score_mean:',df_non_default_svg['svg_score_3'].mean(),\
      'non-default_aes_score_mean:',df_non_default_svg['aes_score_3'].mean(),\
        'combined_score:',df_non_default_svg['combined_score_3'].mean())

non-default_svg_count: 75
non-default_svg_score_mean: 0.03092420762564351 non-default_aes_score_mean: 0.48672293345133455 combined_score: 0.18285711623420722


In [23]:
df['svg_3'].iloc[4]

'<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red"/></svg>'